# Failure Session Labeling Pipeline

This notebook implements a comprehensive pipeline for identifying and labeling failure sessions in solar inverter data.

## Overview
1. **Data Loading**: Load inverter data from parquet files
2. **Data Preprocessing**: Calculate total power and prepare AC power data
3. **Failure Session Identification**: Identify periods when inverters are down while the system is active
4. **Maintenance Session Labeling**: Manually label maintenance sessions
5. **Data Export**: Save labeled failure sessions for further analysis

## Requirements
- Inverter data in parquet format
- Required columns: `event_local_time`, `device_name`, `metric.AC_POWER.MEASURED`


## 1. Import Libraries and Setup


In [33]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
from typing import List, Tuple, Optional
import sys
# Add the project root to Python path
sys.path.append('..')
from inverter_predictive_maintenance.visualize import visualize_failure_timeline

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

print("Libraries imported successfully!")


Libraries imported successfully!


## 2. Configuration and Data Loading


In [4]:
# Configuration
CONFIG = {
    'data_path': '../dataset/inverter_data/',
    'output_path': '../dataset/',
    'min_session_days': 3,  # Minimum duration for a session to be considered
    'power_column': 'metric.AC_POWER.MEASURED',
    'time_column': 'event_local_time',
    'device_column': 'device_name'
}

print(f"Configuration loaded: {CONFIG}")


Configuration loaded: {'data_path': '../dataset/inverter_data/', 'output_path': '../dataset/', 'min_session_days': 3, 'power_column': 'metric.AC_POWER.MEASURED', 'time_column': 'event_local_time', 'device_column': 'device_name'}


In [5]:
def load_inverter_data(data_path: str) -> pd.DataFrame:
    """
    Load inverter data from parquet files.
    
    Args:
        data_path (str): Path to the parquet data directory
        
    Returns:
        pd.DataFrame: Loaded inverter data
        
    Raises:
        FileNotFoundError: If data path doesn't exist
        ValueError: If required columns are missing
    """
    try:
        # Check if path exists
        if not Path(data_path).exists():
            raise FileNotFoundError(f"Data path not found: {data_path}")
        
        # Load data
        print(f"Loading data from: {data_path}")
        inverter_data = pd.read_parquet(data_path)
        
        # Validate required columns
        required_columns = [CONFIG['time_column'], CONFIG['device_column'], CONFIG['power_column']]
        missing_columns = [col for col in required_columns if col not in inverter_data.columns]
        
        if missing_columns:
            raise ValueError(f"Missing required columns: {missing_columns}")
        
        print(f"Data loaded successfully! Shape: {inverter_data.shape}")
        print(f"Columns: {list(inverter_data.columns)}")
        print(f"Date range: {inverter_data[CONFIG['time_column']].min()} to {inverter_data[CONFIG['time_column']].max()}")
        
        return inverter_data
        
    except Exception as e:
        print(f"Error loading data: {e}")
        raise

# Load the data
inverter_data = load_inverter_data(CONFIG['data_path'])
inverter_data.head()


Loading data from: ../dataset/inverter_data/


Data loaded successfully! Shape: (6126272, 59)
Columns: ['event_local_time', 'device_name', 'metric.AC_CURRENT_A.MEASURED', 'metric.AC_CURRENT_B.MEASURED', 'metric.AC_CURRENT_C.MEASURED', 'metric.AC_CURRENT_MAX.MEASURED', 'metric.AC_POWER.MEASURED', 'metric.AC_POWER_LIMIT_SETPOINT.MEASURED', 'metric.AC_VOLTAGE_AB.MEASURED', 'metric.AC_VOLTAGE_BC.MEASURED', 'metric.AC_VOLTAGE_CA.MEASURED', 'metric.AC_VOLTAGE_HI_SETPOINT.MEASURED', 'metric.AC_VOLTAGE_LO_SETPOINT.MEASURED', 'metric.COMM_LINK.MEASURED', 'metric.DC_BATT_VOLTAGE_BUS.MEASURED', 'metric.DC_CURRENT.MEASURED', 'metric.DC_CURRENT_AVG.MEASURED', 'metric.DC_CURRENT_MAX.MEASURED', 'metric.DC_POWER.MEASURED', 'metric.DC_VOLTAGE.MEASURED', 'metric.DC_VOLTAGE_BUS.MEASURED', 'metric.DC_VOLTAGE_N.MEASURED', 'metric.DC_VOLTAGE_P.MEASURED', 'metric.ENERGY_DELIVERED.MEASURED', 'metric.ENERGY_DELIVERED_DAILY.MEASURED', 'metric.ENERGY_DELIVERED_MONTHLY.MEASURED', 'metric.ENERGY_RECEIVED.MEASURED', 'metric.FREQUENCY.MEASURED', 'metric.HEARTBEA

,event_local_time,device_name,metric.AC_CURRENT_A.MEASURED,metric.AC_CURRENT_B.MEASURED,metric.AC_CURRENT_C.MEASURED,metric.AC_CURRENT_MAX.MEASURED,metric.AC_POWER.MEASURED,metric.AC_POWER_LIMIT_SETPOINT.MEASURED,metric.AC_VOLTAGE_AB.MEASURED,metric.AC_VOLTAGE_BC.MEASURED,metric.AC_VOLTAGE_CA.MEASURED,metric.AC_VOLTAGE_HI_SETPOINT.MEASURED,metric.AC_VOLTAGE_LO_SETPOINT.MEASURED,metric.COMM_LINK.MEASURED,metric.DC_BATT_VOLTAGE_BUS.MEASURED,metric.DC_CURRENT.MEASURED,metric.DC_CURRENT_AVG.MEASURED,metric.DC_CURRENT_MAX.MEASURED,metric.DC_POWER.MEASURED,metric.DC_VOLTAGE.MEASURED,metric.DC_VOLTAGE_BUS.MEASURED,metric.DC_VOLTAGE_N.MEASURED,metric.DC_VOLTAGE_P.MEASURED,metric.ENERGY_DELIVERED.MEASURED,metric.ENERGY_DELIVERED_DAILY.MEASURED,metric.ENERGY_DELIVERED_MONTHLY.MEASURED,metric.ENERGY_RECEIVED.MEASURED,metric.FREQUENCY.MEASURED,metric.HEARTBEAT.MEASURED,metric.HW_VERSION.MEASURED,metric.INSUL_MON_AC_RESISTOR.MEASURED,metric.INSUL_MON_DC_RESISTOR.MEASURED,metric.POWER_FACTOR.MEASURED,metric.STATUS_AC_MOD_ADMISSION_TEMP.MEASURED,metric.STATUS_CURRENT_NORMATIVE.MEASURED,metric.STATUS_FAULT_MODULE.MEASURED,metric.STATUS_FAULT_WORD.MEASURED,metric.STATUS_IGBT_MAX_TEMP.MEASURED,metric.STATUS_INSUL_MON_AC.MEASURED,metric.STATUS_INSUL_MON_DC.MEASURED,metric.STATUS_INTERNAL_HUMIDITY.MEASURED,metric.STATUS_INTERNAL_INPUT_WORD.MEASURED,metric.STATUS_INTERNAL_OUTPUT_WORD.MEASURED,metric.STATUS_INTERNAL_TEMP.MEASURED,metric.STATUS_LV_PRESSURE.MEASURED,metric.STATUS_MOD_MAX_TEMP.MEASURED,metric.STATUS_MV_PRESSURE.MEASURED,metric.STATUS_POWER_SOURCE_1.MEASURED,metric.STATUS_POWER_SOURCE_2.MEASURED,metric.STATUS_WARNING_MODULE.MEASURED,metric.STATUS_WARNING_WORD.MEASURED,metric.STATUS_WORD.MEASURED,metric.SVA.MEASURED,metric.SVA_LIMIT_SETPOINT.MEASURED,metric.VAR.MEASURED,metric.VARH_DELIVERED.MEASURED,metric.VARH_DELIVERED_DAILY.MEASURED,metric.VARH_DELIVERED_MONTHLY.MEASURED,metric.VAR_LIMIT_SETPOINT.MEASURED
0,2021-12-02,INV 51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-12-02,INV 52,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-12-02,INV 53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2021-12-02,INV 54,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2021-12-02,INV 55,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Data Preprocessing


In [6]:
def calculate_total_power(inverter_data: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate total power across all inverters for each timestamp.
    
    Args:
        inverter_data (pd.DataFrame): Input inverter data
        
    Returns:
        pd.DataFrame: Total power data with timestamps
    """
    print("Calculating total power...")
    
    # Group by timestamp and sum AC power
    total_power = (
        inverter_data
        .groupby(CONFIG['time_column'])[CONFIG['power_column']]
        .sum()
        .rename('total_power')
        .reset_index()
    )
    
    # Convert to datetime
    total_power[CONFIG['time_column']] = pd.to_datetime(total_power[CONFIG['time_column']])
    
    print(f"Total power calculated for {len(total_power)} timestamps")
    return total_power

# Calculate total power
total_power = calculate_total_power(inverter_data)
total_power.head()


Calculating total power...
Total power calculated for 382892 timestamps


,event_local_time,total_power
0,2021-12-02 00:00:00,0.0
1,2021-12-02 00:05:00,0.0
2,2021-12-02 00:10:00,0.0
3,2021-12-02 00:15:00,0.0
4,2021-12-02 00:20:00,0.0


In [7]:
def prepare_ac_power_data(inverter_data: pd.DataFrame, total_power: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare AC power data with total power information.
    
    Args:
        inverter_data (pd.DataFrame): Input inverter data
        total_power (pd.DataFrame): Total power data
        
    Returns:
        pd.DataFrame: Prepared AC power data
    """
    print("Preparing AC power data...")
    
    # Select relevant columns
    ac_power_data = inverter_data[[
        CONFIG['time_column'], 
        CONFIG['device_column'], 
        CONFIG['power_column']
    ]].copy()
    
    # Rename power column for clarity
    ac_power_data = ac_power_data.rename(columns={
        CONFIG['power_column']: 'inverter_ac_power'
    })
    
    # Fill NaN values with 0
    ac_power_data['inverter_ac_power'] = ac_power_data['inverter_ac_power'].fillna(0)
    
    # Convert to datetime
    ac_power_data[CONFIG['time_column']] = pd.to_datetime(ac_power_data[CONFIG['time_column']])
    
    # Merge with total power data
    ac_power_data = pd.merge(
        ac_power_data, 
        total_power, 
        on=CONFIG['time_column'], 
        how='left'
    )
    
    print(f"AC power data prepared! Shape: {ac_power_data.shape}")
    print(f"Unique devices: {ac_power_data[CONFIG['device_column']].nunique()}")
    
    return ac_power_data

# Prepare AC power data
ac_power_data = prepare_ac_power_data(inverter_data, total_power)
ac_power_data.head()


Preparing AC power data...
AC power data prepared! Shape: (6126272, 4)
Unique devices: 16


,event_local_time,device_name,inverter_ac_power,total_power
0,2021-12-02,INV 51,0.0,0.0
1,2021-12-02,INV 52,0.0,0.0
2,2021-12-02,INV 53,0.0,0.0
3,2021-12-02,INV 54,0.0,0.0
4,2021-12-02,INV 55,0.0,0.0


## 4. Failure Session Identification

### Algorithm Overview

The failure session identification algorithm works as follows:

1. **Iterate Over Each Inverter**: For every unique `device_name` in the dataset, extract its AC power time-series data.

2. **Session Tracking Logic**:
   - **Session Start**: A failure session starts when:
     - `inverter_ac_power == 0` (inverter is down)
     - `total_power > 0` (system is active)
     - No ongoing session exists
   
   - **Session Continuation**: While `inverter_ac_power == 0`, continue updating `end_time`
   
   - **Session End**: When `inverter_ac_power > 0`:
     - If a session is in progress, finalize it
     - Reset session tracking variables

3. **Edge Case Handling**: After processing all timestamps, save any remaining open session.


In [8]:
def identify_failure_sessions(ac_power_data: pd.DataFrame) -> pd.DataFrame:
    """
    Identify failure sessions for each inverter.
    
    A failure session is defined as a period when:
    - An inverter's AC power is 0 (inverter is down)
    - The total system power is > 0 (system is active)
    
    Args:
        ac_power_data (pd.DataFrame): Prepared AC power data
        
    Returns:
        pd.DataFrame: Identified failure sessions with duration
    """
    print("Identifying failure sessions...")
    
    device_names = ac_power_data[CONFIG['device_column']].unique()
    failure_sessions = []
    
    for device_name in device_names:
        print(f"Processing device: {device_name}")
        
        # Get data for this device
        device_data = (
            ac_power_data[ac_power_data[CONFIG['device_column']] == device_name]
            .set_index(CONFIG['time_column'])
            .sort_index()
        )
        
        # Initialize session tracking
        start_time = None
        end_time = None
        
        # Process each timestamp
        for timestamp in device_data.index:
            row = device_data.loc[timestamp]
            
            if row['inverter_ac_power'] == 0:
                # Inverter is down
                if start_time is None and row['total_power'] > 0:
                    # Start a new failure session
                    start_time = timestamp
                end_time = timestamp
            else:
                # Inverter is up
                if start_time is not None:
                    # End the current session
                    failure_sessions.append({
                        'device_name': device_name,
                        'start_time': start_time,
                        'end_time': end_time
                    })
                
                # Reset session tracking
                start_time = None
                end_time = None
        
        # Handle any remaining open session
        if start_time is not None and end_time is not None:
            failure_sessions.append({
                'device_name': device_name,
                'start_time': start_time,
                'end_time': end_time
            })
    
    # Convert to DataFrame
    failure_sessions_df = pd.DataFrame(failure_sessions)
    
    if not failure_sessions_df.empty:
        # Calculate duration
        failure_sessions_df['duration'] = (
            failure_sessions_df['end_time'] - failure_sessions_df['start_time']
        )
        
        # Add duration in days
        failure_sessions_df['duration_days'] = (
            failure_sessions_df['duration'].dt.total_seconds() / (24 * 3600)
        )
    
    print(f"Identified {len(failure_sessions_df)} failure sessions")
    return failure_sessions_df

# Identify failure sessions
failure_sessions = identify_failure_sessions(ac_power_data)
failure_sessions.head()


Identifying failure sessions...
Processing device: INV 51
Processing device: INV 52
Processing device: INV 53
Processing device: INV 54
Processing device: INV 55
Processing device: INV 56
Processing device: INV 57
Processing device: INV 58
Processing device: INV 59
Processing device: INV 60
Processing device: INV 61
Processing device: INV 62
Processing device: INV 63
Processing device: INV 64
Processing device: INV 65
Processing device: INV 66
Identified 26827 failure sessions


,device_name,start_time,end_time,duration,duration_days
0,INV 51,2021-12-02 07:00:00,2021-12-02 07:00:00,0 days 00:00:00,0.000000
1,INV 51,2021-12-02 07:10:00,2021-12-02 07:10:00,0 days 00:00:00,0.000000
2,INV 51,2021-12-03 07:00:00,2021-12-03 07:00:00,0 days 00:00:00,0.000000
3,INV 51,2021-12-04 07:15:00,2021-12-04 07:25:00,0 days 00:10:00,0.006944
4,INV 51,2021-12-04 16:30:00,2021-12-05 07:55:00,0 days 15:25:00,0.642361


In [9]:
# Display summary statistics
if not failure_sessions.empty:
    print("\n=== Failure Sessions Summary ===")
    print(f"Total sessions: {len(failure_sessions)}")
    print(f"Unique devices: {failure_sessions['device_name'].nunique()}")
    print(f"\nDuration statistics:")
    print(failure_sessions['duration_days'].describe())
    
    print("\nSessions by device:")
    device_counts = failure_sessions['device_name'].value_counts()
    print(device_counts)
else:
    print("No failure sessions identified.")



=== Failure Sessions Summary ===
Total sessions: 26827
Unique devices: 16

Duration statistics:
count    26827.000000
mean         0.191632
std          1.009983
min          0.000000
25%          0.000000
50%          0.003472
75%          0.399306
max         94.697917
Name: duration_days, dtype: float64

Sessions by device:
device_name
INV 59    2139
INV 62    1929
INV 64    1867
INV 60    1827
INV 56    1816
INV 63    1807
INV 51    1766
INV 53    1759
INV 57    1683
INV 61    1682
INV 66    1656
INV 55    1614
INV 52    1541
INV 58    1365
INV 54    1261
INV 65    1115
Name: count, dtype: int64


## 5. Save Initial Failure Sessions


In [10]:
def save_failure_sessions(failure_sessions: pd.DataFrame, output_path: str) -> None:
    """
    Save failure sessions to CSV file.
    
    Args:
        failure_sessions (pd.DataFrame): Failure sessions data
        output_path (str): Output directory path
    """
    try:
        # Ensure output directory exists
        Path(output_path).mkdir(parents=True, exist_ok=True)
        
        # Save to CSV
        file_path = Path(output_path) / 'failure_sessions.csv'
        failure_sessions.to_csv(file_path, index=False)
        
        print(f"Failure sessions saved to: {file_path}")
        print(f"File size: {file_path.stat().st_size / 1024:.2f} KB")
        
    except Exception as e:
        print(f"Error saving failure sessions: {e}")
        raise

# Save failure sessions
save_failure_sessions(failure_sessions, CONFIG['output_path'])


Failure sessions saved to: ..\dataset\failure_sessions.csv
File size: 2006.98 KB


## 6. Filter Long-Duration Sessions

Filter sessions to keep only those longer than the minimum duration threshold.


In [36]:
from inverter_predictive_maintenance.preprocess import load_failure_sessions
# Filter long sessions
long_failure_sessions = load_failure_sessions(CONFIG['output_path']+'failure_sessions.csv')
long_failure_sessions.head()


Kept 61 sessions longer than 3 days


,device_name,start_time,end_time,duration,duration_days,maintenance
904,INV 51,2023-03-17 17:10:00,2023-03-21 12:20:00,3 days 19:10:00,3.798611,False
1271,INV 51,2024-04-25 12:55:00,2024-04-30 08:00:00,4 days 19:05:00,4.795139,False
1653,INV 51,2025-04-20 13:35:00,2025-05-23 15:00:00,33 days 01:25:00,33.059028,False
2186,INV 52,2022-11-07 06:35:00,2022-11-10 10:05:00,3 days 03:30:00,3.145833,False
3086,INV 52,2025-03-29 13:55:00,2025-04-04 12:30:00,5 days 22:35:00,5.940972,False


## 7. Prepare for Manual Labeling

Add maintenance labels and session IDs for manual labeling.


In [37]:
def prepare_for_labeling(failure_sessions: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare failure sessions for manual maintenance labeling.
    
    Args:
        failure_sessions (pd.DataFrame): Filtered failure sessions
        
    Returns:
        pd.DataFrame: Sessions ready for labeling
    """
    if failure_sessions.empty:
        return failure_sessions
    
    print("Preparing sessions for manual labeling...")
    
    # Reset index
    labeled_sessions = failure_sessions.reset_index(drop=True)
    
    # Add maintenance label column
    labeled_sessions['maintenance'] = False
    
    # Add session ID
    labeled_sessions['session_id'] = range(len(labeled_sessions))
    
    print(f"Prepared {len(labeled_sessions)} sessions for labeling")
    
    return labeled_sessions

# Prepare for labeling
labeled_sessions = prepare_for_labeling(long_failure_sessions)
labeled_sessions.head()


Preparing sessions for manual labeling...
Prepared 61 sessions for labeling


,device_name,start_time,end_time,duration,duration_days,maintenance,session_id
0,INV 51,2023-03-17 17:10:00,2023-03-21 12:20:00,3 days 19:10:00,3.798611,False,0
1,INV 51,2024-04-25 12:55:00,2024-04-30 08:00:00,4 days 19:05:00,4.795139,False,1
2,INV 51,2025-04-20 13:35:00,2025-05-23 15:00:00,33 days 01:25:00,33.059028,False,2
3,INV 52,2022-11-07 06:35:00,2022-11-10 10:05:00,3 days 03:30:00,3.145833,False,3
4,INV 52,2025-03-29 13:55:00,2025-04-04 12:30:00,5 days 22:35:00,5.940972,False,4


## 8. Visualization for Manual Labeling

Create visualizations to help with manual maintenance session identification.


In [38]:
labeled_sessions.loc[[5-2, 19-2, 41-2, 52-2], "maintenance"] = True
labeled_sessions.loc[[6, 19, 45], "maintenance"] = True
labeled_sessions.loc[[24, 34, 42, 53], "maintenance"] = True
labeled_sessions.loc[[0, 27, 32, 36, 46], "maintenance"] = True
labeled_sessions.loc[[2, 29, 38, 47, 16], "maintenance"] = True
labeled_sessions.loc[[43, 56, 60], "maintenance"] = True
labeled_sessions.loc[[33, 41, 51], "maintenance"] = True
labeled_sessions.loc[[5, 23], "maintenance"] = True
labeled_sessions.loc[[15, 55], "maintenance"] = True

In [ ]:
visualize_failure_timeline(labeled_sessions)

In [44]:
labeled_sessions.drop(columns=['duration_days'], inplace=True)
labeled_sessions.to_csv(r"../dataset/failure_sessions_w_maintenance.csv", index=False)